In [1]:
install.packages("data.table")
install.packages("dplyr")
library(data.table)
library(dplyr)
library(VectorForgeML)

Installing package into 'C:/Users/mushe/AppData/Local/R/win-library/4.5'
(as 'lib' is unspecified)



package 'data.table' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\mushe\AppData\Local\Temp\RtmpeEy3m9\downloaded_packages


Installing package into 'C:/Users/mushe/AppData/Local/R/win-library/4.5'
(as 'lib' is unspecified)



package 'dplyr' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\mushe\AppData\Local\Temp\RtmpeEy3m9\downloaded_packages



Attaching package: 'dplyr'


The following objects are masked from 'package:data.table':

    between, first, last


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




In [2]:
df <- fread("Dataset/combined_dataset.csv", data.table = FALSE)

df_balanced <- df %>%
  group_by(Label) %>%
  sample_n(min(250000, n())) %>%
  ungroup()

names(df_balanced) <- trimws(names(df_balanced))


# Convert label properly
y <- as.factor(df_balanced$Label)
y <- as.numeric(y) - 1

X <- df_balanced
X$Label <- NULL

# Replace infinite values
X[sapply(X, is.numeric)] <- lapply(X[sapply(X, is.numeric)], function(col){
  col[is.infinite(col)] <- NA
  col
})

# Remove NA rows safely
data <- cbind(X, Label=y)
data <- na.omit(data)

y <- data$Label
X <- data
X$Label <- NULL

Warning message in require_bit64_if_needed(ans):
"Some columns are type 'integer64' but package bit64 is not installed. Those columns will print as strange looking floating point data. There is no need to reload the data. Simply install.packages('bit64') to obtain the integer64 print method and print the data again."


In [3]:
install.packages("VectorForgeML")
library(VectorForgeML)

Warning message:
"package 'VectorForgeML' is in use and will not be installed"


In [4]:
ls("package:VectorForgeML")

[1] "accuracy_score"        "ColumnTransformer"     "confusion_matrix"     
 [4] "confusion_stats"       "DecisionTree"          "drop_constant_columns"
 [7] "f1_score"              "find_best_k"           "fit_linear_model"     
[10] "KMeans"                "KNN"                   "LabelEncoder"         
[13] "LinearRegression"      "LogisticRegression"    "macro_f1"             
[16] "macro_precision"       "macro_recall"          "MinMaxScaler"         
[19] "mse"                   "OneHotEncoder"         "PCA"                  
[22] "Pipeline"              "plot_confusion_matrix" "precision_score"      
[25] "predict_linear_model"  "r2_score"              "RandomForest"         
[28] "recall_score"          "RidgeRegression"       "rmse"                 
[31] "SoftmaxRegression"     "StandardScaler"        "train_test_split"

In [5]:

cat_cols <- names(X)[sapply(X, is.character)]
num_cols <- names(X)[!sapply(X, is.character)]

X[cat_cols] <- lapply(X[cat_cols], as.factor)

split <- train_test_split(X, y, 0.2, 42)

pre <- ColumnTransformer$new(
  num_cols, cat_cols,
  StandardScaler$new(),
  OneHotEncoder$new()
)

pipe <- Pipeline$new(list(
  pre,
  RandomForest$new(ntrees=100, max_depth=7, 4, mode="classification")
))

pipe$fit(split$X_train, split$y_train)

pred <- pipe$predict(split$X_test)

pred <- ifelse(pred > 0.5, 1, 0)

cat("Accuracy:", accuracy_score(split$y_test, pred), "\n")

Accuracy: 0.3041144 
